In [9]:
# import tensorflow
# tensorflow.__version__

# import numpy
# numpy.__version__

# import sklearn
# sklearn.__version__

# import matplotlib
# matplotlib.__version__

import scikeras
scikeras.__version__

'0.13.0'

In [ ]:

# https://stackoverflow.com/questions/55178230/what-is-the-difference-between-keras-and-tf-keras
'''
The difference between tf.keras and keras is the Tensorflow specific enhancement to the framework.
keras is an API specification that describes how a Deep Learning framework should implement certain part, related to the model definition and training. Is framework agnostic and supports 
different backends (Theano, Tensorflow, ...)
tf.keras is the Tensorflow specific implementation of the Keras API specification.
It adds the framework the support for many Tensorflow specific features like: perfect support for tf.data.Dataset as input objects, support for eager execution, ...
In Tensorflow 2.0 tf.keras will be the default and I highly recommend to start working using tf.keras
'''

In [1]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error, mean_absolute_error, mean_squared_log_error

import sys
from sklearn.linear_model import LinearRegression

import timeit
import tensorflow as tf   # in tf.keras.utils.set_random_seed(seed)

# grid search
from scikeras.wrappers import  KerasRegressor
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.models import Sequential   # in grid.fit(A_train, b_train)
from tensorflow.keras.layers import Dense






# put function in separate file after 
# =========================================================================================================================
                   
def plot_test_data(true_data_test, pred_data_test, start_test, end_test, iteration, title_name ):
    """ 
    Plotting all test data

    Parameters:
        true_data_test  : true data of the test set
        pred_data_test  : prediction data of the test set
        start_test      : start from points defined    
        end_test        : end at points defined
        iteration       : list elements of the column_name
        title_name      : list of title name form column_names.py
    
    Returns:
        -
    """
    plt.rcParams['figure.figsize'] = [22, 4]
    plt.rcParams.update({'font.size': 8})

    
    fig = plt.figure()
    ax1 = fig.add_subplot(121)
    
    
    plt.plot(true_data_test[start_test:end_test], '-x', color='k', linewidth=1, ms=4, label='True data') # True relationship
    plt.plot(pred_data_test[start_test:end_test], 'o', color='r', linewidth=1, ms=3, label='Regression predict')
    plt.xlabel('time step')
    plt.ylabel('value')
    plt.title('Test set ' + title_name[iteration])
    # plt.title(f'Train {iteration}')
    plt.legend()
    

    # save plot?




def plot_train_data(true_data_train, pred_data_train, start_train, end_train, iteration, title_name):
    """ 
    Plotting all train data

    Parameters:
        true_data_train  : true data of the train set
        pred_data_train  : prediction data of the test set
        start_train      : start from points defined    
        end_train        : end at points defined
        iteration        : list elements of the column_name
        title_name       : list of title name form column_names.py
    
    Returns:
        -
    """
    plt.rcParams['figure.figsize'] = [22, 4]
    plt.rcParams.update({'font.size': 8})
    
    fig = plt.figure()
    ax1 = fig.add_subplot(121)
    
    
    plt.plot(true_data_train[start_train : end_train], '-x', color='k', linewidth=1, ms=4, label='True value') # True relationship
    plt.plot(pred_data_train[start_train : end_train], 'o', color='r', linewidth=1, ms=3, label='Prediction')
    plt.xlabel('time step')
    plt.ylabel('value')
    plt.title('Train set ' + title_name[iteration])
    # plt.title(f'Train {iteration}')
    plt.legend()

    # save plot?

def plot_partial_data(percentage, true_data, pred_data, iteration, title_name):
    """ 
    Instead of plotting full data, select fornt part and end part of data to plot

    Parameters:
        percentage  : select 10 %; 20%; ... of front or back graph 
        true_data   : true data of train / test set
        pred_data   : prediction data of train/ test set
        iteration   : list elements of the column_name
        title_name  : list of title name form column_names.py

    Returns:
        -
    """
    plt.rcParams['figure.figsize'] = [11, 4]
    plt.rcParams.update({'font.size': 8})
    
    
         
    front_part = true_data.shape[0] * percentage // 100
    back_part = - true_data.shape[0] * percentage // 100 # count from back

    # front plot
    fig = plt.figure()
    ax1 = fig.add_subplot(121)   # Three integers (nrows, ncols, index).
    plt.plot(true_data[: front_part], '-x', color='k', linewidth=1, ms=4, label='True value') # True relationship
    plt.plot(pred_data[: front_part], 'o', color='r', linewidth=1, ms=3, label='Predction')
    plt.xlabel('time step')
    plt.ylabel('value')
    plt.title(f'first {percentage} % ' + title_name[iteration])
    # plt.title(f'Train {iteration}')
    plt.legend()



    # back plot
    ax2 = fig.add_subplot(122)
    plt.plot(true_data[back_part :], '-x', color='k', linewidth=1, ms=4, label='True value') # True relationship
    plt.plot(pred_data[back_part :], 'o', color='r', linewidth=1, ms=3, label='Prediction')
    
    # plt.xticks( np.arange(true_data.shape[0]) , np.arange( true_data.shape[0] + back_part, true_data.shape[0], step = 100))   # xticks(np.arange(3), ['Tom', 'Dick', 'Sue'])  # Set text labels.
    plt.xlabel('time step')
    
    plt.ylabel('value')
    plt.title(f'last {percentage} % ' + title_name[iteration])
    # plt.title(f'Train {iteration}')
    plt.legend()

    
    

def performance_metrics(true_result, pred_result): # 
    """ 
    Calculate performance metrics.

    Parameters:
        true_result : true data in dataset
        pred_result : prediction data

    Returns:
        column_099  : column names that with R^2 < 0.99
    """
    
    # R2
    r2_uni_avg = r2_score(true_result, pred_result, multioutput='uniform_average')
    r2_raw_val = r2_score(true_result, pred_result, multioutput='raw_values' )
    r2_raw_val2 = [f"{x:.4f}" for x in r2_raw_val]
    r2_var_wei = r2_score(true_result, pred_result, multioutput='variance_weighted')

    # mean square
    mse = mean_squared_error(true_result, pred_result)
    mse_raw = mean_squared_error(true_result, pred_result, multioutput='raw_values' )
    rmse = np.sqrt(mse)

    # mean absolute 
    mae = mean_absolute_error(true_result, pred_result)
    mape = mean_absolute_percentage_error(true_result, pred_result, multioutput='uniform_average')
    mape_raw = mean_absolute_percentage_error(true_result, pred_result, multioutput='raw_values')
    mape_raw = [x * 100 for x in mape_raw]
    mape_raw = [f"{x :.4f} %"  for x in mape_raw]

    # The MAPE formula here does not represent the common “percentage” definition: the percentage in the range [0, 100] 
    # is converted to a relative value in the range [0, 1] by dividing by 100. Thus, an error of 200% corresponds to a relative error of 2. 
    # o obtain the mean absolute percentage error as per the Wikipedia formula, multiply the mean_absolute_percentage_error computed here by 100.    

    
    
    # ...........Print evaluation metrics...........

    # R2
    print(f"R-squared (uniform average)  : {r2_uni_avg:.4f}")  # f"{value:e}"   # For scientific notation
    print(f"R-squared (raw values)       : ")
    # print(r2_raw_val2)                        # < ------------------------ raw value
    print("Number of R2 < 0.99: ",  sum(i < 0.99 for i in r2_raw_val), '/', len(r2_raw_val))
    # print(f"R-squared (variance weighted): {r2_var_wei:.4f}")
    #if i in r2_raw_val < 0.99:
    column_099 = np.where(r2_raw_val < 0.99)
    # print(column_099)
    # for i in r2_raw_val:
    #     if i < 0.99:
    #         print(i) # 
            
    
    print()

    # mean square
    print(f"Mean squared error (avg): {mse:.4e}")
    # print(f"Mean squared error (log): {-np.log(mse)}")
    print(f"Mean squared error (raw):")
    # print(mse_raw)                            # < ------------------ raw value
    print(f"Root mean squared error: {rmse:.4e}")
    # print(f"Root mean squared error (log): {-np.log(rmse)}")
    
    
    # Mean absolute
    print(f"Mean absolute error (avg): {mae:.4e}")
    print(f"Mean absolute percentage error MAPE (avg):{mape * 100:.4e} %")
    print(f"Mean absolute percentage error MAPE (raw):")
    # print(mape_raw)                                       # < -------------- raw value

    # find out maximum number of raw values
    return column_099



# remove not good performance metrics
# print(f"Mean square log error (avg): {msle:.4e}")  # Mean Squared Logarithmic Error cannot be used when targets contain negative values.
# msle = mean_squared_log_error(true_result, pred_result) # Mean Squared Logarithmic Error cannot be used when targets contain negative values.
    # acc_score = []
    # # scoring
    # for i in range(true_result.shape[1]):
    #     acc_score[i] = accuracy_score(true_result[i], pred_result[i])   # only for classification 
    # scoring
    # for i in range(true_result.shape[1]):
    #     print(f"Accuracy:", acc_score)
    
# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++




def read_files(molecule):
    """ 
    read and load the data based on molecule selected

    Parameters:
        molecule: type of moelcule selected 

    Returns:
        data_density  : flattened density matrices
        data_myfock   : flattened fock matrices
    """
    # --------------------- load full dataset ------------------------------
    
    if molecule == 'H2-sto3g-10000':
        print("Current dataset: H2-sto3g-10000")
        data_density = np.loadtxt("densitylist-Copy1(H2)-sto3g-10000.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(H2)-sto3g-10000.csv", delimiter = ',')
    
    elif molecule == 'LiH-sto3g-10000':
        print("Current dataset: LiH-sto3g-10000")
        data_density = np.loadtxt("densitylist-Copy1(LiH)-sto3g-10000.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(LiH)-sto3g-10000.csv", delimiter = ',')
    
    
    elif molecule == 'H2O-sto3g-10000':
        print("Current dataset: H2O-sto3g-10000")
        data_density = np.loadtxt("densitylist-Copy1(H2O)-sto3g-10000.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(H2O)-sto3g-10000.csv", delimiter = ',')

    elif molecule == 'H2-sto3g-20000':
        print("Current dataset: H2-sto3g-20000")
        data_density = np.loadtxt("densitylist-Copy1(H2)-sto3g-20000.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(H2)-sto3g-20000.csv", delimiter = ',')

    elif molecule == 'LiH-sto3g-20000':
        print("Current dataset: LiH-sto3g-20000")
        data_density = np.loadtxt("densitylist-Copy1(LiH)-sto3g-20000.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(LiH)-sto3g-20000.csv", delimiter = ',')

    elif molecule == 'H2O-sto3g-20000':
        print("Current dataset: H2O-sto3g-20000")
        data_density = np.loadtxt("densitylist-Copy1(H2O)-sto3g-20000.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(H2O)-sto3g-20000.csv", delimiter = ',')

    elif molecule == 'LiH-631g-10000':
        print("Current dataset: LiH-631g-10000")
        data_density = np.loadtxt("densitylist-Copy1(LiH)-631g-10000.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(LiH)-631g-10000.csv", delimiter = ',')

    elif molecule == 'H2O-631g-10000':
        print("Current dataset: H2O-631g-10000")
        data_density = np.loadtxt("densitylist-Copy1(H2O)-631g-10000.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(H2O)-631g-10000.csv", delimiter = ',')

    elif molecule == 'LiH-sto3g-10000-y':
        print("Current dataset: LiH-sto3g-10000-y")
        data_density = np.loadtxt("densitylist-Copy1(LiH)-sto3g-10000-y.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(LiH)-sto3g-10000-y.csv", delimiter = ',')

    elif molecule == 'LiH-sto3g-10000-x':
        print("Current dataset: LiH-sto3g-10000-x")
        data_density = np.loadtxt("densitylist-Copy1(LiH)-sto3g-10000-x.csv", delimiter = ',')
        data_myfock = np.loadtxt("myfocklist-Copy1(LiH)-sto3g-10000-x.csv", delimiter = ',')
    
    else:
        print('No dataset selected!')
        sys.exit()
    return data_density, data_myfock





def create_model(molecule, sample_size, train_percentage, scaler_type, remove_kick, kick_size, plot_training_data, plot_testing_data, 
                      start_train, end_train, start_test, end_test, column_name, plot_front_end, plot_front_end_percentage, 
                      perform_metric_front_end, performance_metric_front_end_percentage, scikit_LR_option,
                      check_noise, filter_noise,
                 Neural_Network_option, grid_search_option 
                      ):
    """ 
    Perform machine learning based on parameters selected

    Parameters:
         molecule  (str)          : type of moleucle (dataset) selected
         sample_size (int)        : sample size is 10000 or 20000
         train_percentage (int / list) : how many percentage of training data  
         scaler_type      (str)   : define types of scaler used
         remove_kick      (bool)  : remove delta kick values or not
         kick_size          (int) : define delta kick values datapoints to remove
         plot_training_data (bool) : plotting all training data? 
         plot_testing_data   (bool): plotting all testing data? 
         start_train (int)  :    start plotting train set from points defined
         end_train   (int)  :    end plotting train set at points defined
         start_test  (int)  :    start plotting test set from points defined
         end_test    (int)  :    end plotting test set at points defined
         column_name (list) :    element name of the density matrices / fock matrices for plotting from column_name.py file
         plot_front_end           (bool) :  plot partial front part and end part of graph? 
         plot_front_end_percentage (int) :  plot how many percentage of front / end graph
         perform_metric_front_end (bool) :  calculate performance metrics for front part/ end part of data
         performance_metric_front_end_percentage (int) :  include how many percentage of front / end part of data for performance metrics
         scikit_LR_option (bool) :    Perform linear regression? 
         check_noise      (bool) :    check are there noise (zeros) in the data? 
         filter_noise     (bool) :    filter out the noise (zeros) in the data? 
         Neural_Network_option (bool) : Perform Neural Network?
         grid_search_option    (bool) : Perform Neural Network with grid search?  

    Returns:
        (options - to check data after calculation)
        B_result_train_true : true data from train set
        B_result_train_pred : prediciton from train set
        B_result_test_true  : true data from test set
        B_result_test_pred  : predictio data from test set
    """

    
    print("+++++++++++++++++++++++++++ START ++++++++++++++++++++++++++++++++")

    if remove_kick == False:
        print("kick values are included.(full data)")
        print("train / test percentage: ", train_percentage ,"% / ", 100 - train_percentage, "%" )
        print("sample_size is:", sample_size)
        
        training_size = round( (sample_size) * train_percentage / 100)
        print("training_size is:", training_size)
        
        test_size = sample_size - training_size
        print("test_size is:", test_size)
    else:
        print("removed kick values.")
        print("train / test percentage: ", train_percentage ,"% / ", 100 - train_percentage, "%" )
        print("sample_size is:", sample_size - kick_size)
        
        training_size = round( (sample_size - kick_size ) * train_percentage / 100)
        print("training_size is:", training_size)
        
        test_size = sample_size - kick_size - training_size 
        print("test_size is:", test_size)
        
    
    # load datafiles 
    data_density, data_myfock = read_files(molecule)
    


    # --------------------- find out which columns are noise data -------------------
    threshold = 1E-13

    if check_noise == True:
        mean_row_density = np.mean(np.abs(data_density), axis=0) # abs not mean
        print("length mean row:", len(mean_row_density))
        # print("Mean across features density matrix: ", mean_row)
        for i in range(len(mean_row_density)):
            # print("i is: ", i)
            if mean_row_density[i] < threshold:   # if abs(mean_row_density[i]) < 1E-13
                print(f"those density features with mean < {threshold}: {i} --> {mean_row_density[i]:.4e} {column_name[i]}")
    
        print()
        
        mean_row_fock = np.mean(np.abs(data_myfock), axis=0)
        print("length mean row:", len(mean_row_fock))
        # print("Mean across features density matrix: ", mean_row)
        for i in range(len(mean_row_fock)):
            # print("i is: ", i)
            if mean_row_fock[i] < threshold:     # if abs(mean_row_fock[i]) < 1E-13
                print(f"those fock features with mean < {threshold}: {i} --> {mean_row_fock[i]:.4e} {column_name[i]}")

    else:
        pass

    if filter_noise == True:
        mean_row_density = np.mean(np.abs(data_density), axis=0)
        mean_row_fock =    np.mean(np.abs(data_myfock), axis=0)
         # np.delete(array , row / colmn index to delete , axis = 1 / 0 )
        index_location_density = np.where( mean_row_density  < threshold)
        index_location_fock =    np.where( mean_row_fock  < threshold)
        
        # data_density[:,index_location_density] = 0
        # data_myfock[:,index_location_fock] = 0

       
        data_density = np.delete(data_density, index_location_density , 1 ) 
        data_myfock  = np.delete(data_myfock , index_location_fock , 1 )  
        
        #print("Replaced density / Fock with true zero value")
        print("density shape after filter out zeros: ", data_density.shape)
        print("fock    shape after filter out zeros: ", data_myfock.shape)
        
        print("len column name bf:", len(column_name))
        column_name = np.delete(column_name, index_location_fock)
        print("len column name af:", len(column_name))
    
    # ------------------- select types of scaler --------------------------

    if Neural_Network_option == True:
        if scaler_type == 'MinMax':
            scalar = MinMaxScaler()
            print('Using MimMax Scaler')
            data_density = scalar.fit_transform(data_density)
            A = data_density 
            B = scalar.fit_transform(data_myfock)
        
        elif scaler_type == 'Std':
            scalar = StandardScaler()
            print('Using Std Scaler')
            data_density = scalar.fit_transform(data_density)
            A =  data_density  # attach ones to A for intercept
            B =  scalar.fit_transform(data_myfock)
        
        else:
            print('No scaler')
            A = data_density 
            B = data_myfock

    else:
        if scaler_type == 'MinMax':
            scalar = MinMaxScaler()
            print('Using MinMax Scaler')
            data_density = scalar.fit_transform(data_density)
            
            if scikit_LR_option == True:
                A = data_density
            else: 
               pass
    
            B =  scalar.fit_transform(data_myfock)
        
        elif scaler_type == 'Std':
            scalar = StandardScaler()
            print('Using Std Scaler')
            data_density = scalar.fit_transform(data_density)
            if scikit_LR_option == True:
                A = data_density
            else: 
                pass
            B =  scalar.fit_transform(data_myfock)
        
        else:
            print('No scaler used')
            
            if scikit_LR_option == True:
                A = data_density
            else: 
                pass
            B =  data_myfock
    
    
    
    print()

    # --------------- Set delta kick skip points ---------------------

    if remove_kick == False: # remove kick False - take whole sample
        print("delta kick sample is included, take sample from first (whole dataset)")
        A = A[0 : sample_size]
        # print("A shape whole: ", A.shape)
        B = B[0 : sample_size]
        # print("B shape whole: ", B.shape)
        test_size = sample_size - training_size   # test size
        
    else:   # remove_kick == True, remove first part of sample
        print("delta kick sample is excluded, take sample exclude first n values")
        A = A[kick_size : sample_size ]
        # print("A shape removed kick: ", A.shape)
        B = B[kick_size : sample_size ]
        # print("B shape removed kick: ", B.shape)
        test_size = sample_size - training_size - kick_size  # test size
        
   

    
    # B_result_true = np.empty([n , B.shape[1]])
    B_result_train_true = np.empty([ training_size , B.shape[1]])  # initizlize empty array, to attach result back into array for inverse transform
    B_result_train_pred = np.empty([ training_size , B.shape[1]])
    print(" B_result_train_true shape",  B_result_train_true.shape)
    print(" B_result_train_pred shape",  B_result_train_pred.shape)
    
    B_result_test_true = np.empty([ test_size , B.shape[1]])  # initizlize empty array, to attach result back into array for inverse transform
    B_result_test_pred = np.empty([ test_size , B.shape[1]])
    print(" B_result_test_true shape",  B_result_test_true.shape)
    print(" B_result_test_pred shape",  B_result_test_pred.shape)
    
    
    if plot_training_data == True:
            print('Plotting training data is on')
    else:
            print('Plotting training data is off')
    
        # plot testing data
    if plot_testing_data == True:
            print('Plotting testing data is on')
    else:
            print('Plotting testing data is off')
        
    print("B.shape[1] is number of dimension:", B.shape[1] )


    # ----------------------- run svd / LR regression on each pred column ---------------------------------
    if  scikit_LR_option == True and Neural_Network_option == True:
        print("not choosing ML method correctly!")
        sys.exit()
  
    elif Neural_Network_option == True:
        print()
        print(".............Solving using Neural-Network.............")
        
    elif scikit_LR_option == True:
        print()
        print(".............Solving using Scikit-learn LR regression..........")
   
    else:
        print()
        print(".............. not choosing ML method correctly! ...........")
        sys.exit()

    start_ML = timeit.default_timer()
    
    if Neural_Network_option == True:

        # define common parameter for grid search and NN
        num_epoch = 50
        num_batch_size = 64
        
        start_grid = timeit.default_timer()
        seed = 42
        tf.keras.utils.set_random_seed(seed)   #  make almost any Keras program fully deterministic
        
        if grid_search_option == True:
            b = B
            b_train = b[0:training_size]  # take all training value until training size
            A_train = A[0:training_size]
        
            # use second half test the model
            b_test = b[training_size: sample_size]   # take testing value
            A_test = A[training_size: sample_size]

            print("-------Performing grid search-------")
            # Function to create model, required for KerasClassifier
            def create_model(hidden_layers,opt, neuron):
            
                model = Sequential()
                model.add(tf.keras.layers.InputLayer(shape=(A_train.shape[1],))) 
                # model.add(Dense(256, input_dim = A_train.shape[1], activation='relu'))  # initialize input layer
            
            
                for i in range(hidden_layers): # https://stackoverflow.com/questions/47788799/grid-search-the-number-of-hidden-layers-with-keras
                    # add one hidden layer
                    if hidden_layers == 0:
                        print("no hidden layers")
                        pass
                    model.add(Dense(neuron, activation = 'relu'))
                # model.add(Dropout(dropout_rate))
                model.add(Dense(A_train.shape[1], kernel_initializer='uniform', activation='linear')) # Initializers define the way to set the initial random weights of Keras layers.
                # https://machinelearningmastery.com/grid-search-hyperparameters-deep-learning-models-python-keras/
               
                # opt = keras.optimizers.Adam(learning_rate=0.01)
                # adam = tf.keras.optimizers.Adam(learning_rate=0.01)   # learning rate
                # sgd =  tf.keras.optimizers.SGD(learning_rate=0.01)
                
                model.compile(loss='mean_squared_error',
                              optimizer= opt, 
                              metrics=['mse']) #note: metrics could also be 'mse'
                return model

            # create model
            model = KerasRegressor(model=create_model, 
                                   epochs = num_epoch, 
                                   batch_size = num_batch_size, 
                                   verbose=1)
    
            # define the grid search parameters
         
            # layers
            hidden_layers = [0, 1, 2]     

            # optimizer
            optimizer = ['adam', 'sgd' ] # 'aa'] --> random optimizer will not work
            
            # number neuron
            neuron_num = [32, 64, 128, 256]
            
            param_grid = dict( 
                model__hidden_layers = hidden_layers,
                model__opt = optimizer, 
                model__neuron = neuron_num
                              
                              )  # in the SciKeras wrapper, you will route the parameters to the optimizer with the prefix optimizer__.
            
            
            
            grid = GridSearchCV(estimator=model, 
                                param_grid=param_grid,
                                n_jobs= -1, 
                                cv=3, 
                                verbose = 1)
            
            grid_result = grid.fit(A_train, b_train)
            
            
            # summarize the results
            print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
            print()
            
            means = grid_result.cv_results_['mean_test_score']    # R2 mean result
            stds = grid_result.cv_results_['std_test_score']
            params = grid_result.cv_results_['params']
            # for mean, stdev, param in zip(means, stds, params):
            #     print("%f (%f) with: %r" % (mean, stdev, param))
            
            
            
            end_grid = timeit.default_timer()
            print('Time grid %8.3f s'%(end_grid - start_grid))

            best_hidden_layers = grid_result.best_params_['model__hidden_layers']
            best_neuron = grid_result.best_params_['model__neuron']
            best_optimizer = grid_result.best_params_['model__opt']

        else:
            print("Not performing grid search, only neural network")
            pass
        
        
        adam = tf.keras.optimizers.Adam(learning_rate=0.01)   # learning rate
        sgd =  tf.keras.optimizers.SGD(learning_rate=0.01) 
        # parameters that can be adjusted
        
        if grid_search_option == True:
            print("Using grid search best parameters")
            neuron =  best_neuron
            optimizer_select  = best_optimizer
            hidden_layer =  best_hidden_layers
        
        else:    # default value
            print("Using default neural network parameters")
            neuron = 256
            optimizer_select = adam
            hidden_layer = 0
        
       
        # 
        ## Perform NN regression 
        b = B
        b_train = b[0:training_size]  # take all training value until training size
        A_train = A[0:training_size]
    
        # use second half test the model
        b_test = b[training_size: sample_size]   # take testing value
        A_test = A[training_size: sample_size]


        
        
        
        print("A_train.shape[1]:", A_train.shape[1], A_train.shape)
        print("b_train.shape[1]:", b_train.shape[1], b_train.shape)
        print("A_test.shape[1]:", A_test.shape[1], A_test.shape)
        print("b_test.shape[1]:", b_test.shape[1], b_test.shape)
        # https://stats.stackexchange.com/questions/181/how-to-choose-the-number-of-hidden-layers-and-nodes-in-a-feedforward-neural-netw
       
        model = tf.keras.models.Sequential()
        model.add(tf.keras.layers.InputLayer(shape=(A_train.shape[1],))) # input layer should be number of features 

        if hidden_layer == 0:
            print("no hidden layer used")
        else:
            # for i in range(hidden_layers): 
            #     model.add(tf.keras.layers.Dense(neuron, activation='relu')) 
            #     print(f"hidden layer {i}")
            pass
            
        # # set default as 0 hidden layer                                # hidden layer 0
        # model.add(tf.keras.layers.Dense(256, activation='relu'))       # hidden layer 1 
        # model.add(tf.keras.layers.Dense(256, activation='relu'))       # hidden layer 2
        # model.add(tf.keras.layers.Dense(256, activation='relu'))       # hidden layer 3
        # model.add(tf.keras.layers.Dense(256, activation='relu'))       # hidden layer 4
        # model.add(tf.keras.layers.Dense(256, activation='relu'))       # hidden layer 5

        # Try use different number of neurons                               # hidden layer 0
        # model.add(tf.keras.layers.Dense(4, activation='relu'))       # hidden layer 1 
        # model.add(tf.keras.layers.Dense(8, activation='relu'))       # hidden layer 2
        # model.add(tf.keras.layers.Dense(32, activation='relu'))       # hidden layer 3
        # model.add(tf.keras.layers.Dense(64, activation='relu'))       # hidden layer 4
        # model.add(tf.keras.layers.Dense(128, activation='relu'))       # hidden layer 5
        # model.add(tf.keras.layers.Dense(256, activation='relu'))       # hidden layer 5
        # model.add(tf.keras.layers.Dense(512, activation='relu'))       # hidden layer 5
        
        model.add(tf.keras.layers.Dense(A_train.shape[1], activation='linear'))     
        
        model.compile(loss='mean_squared_error', 
                    optimizer = optimizer_select, 
                    metrics=['mse'])
        A_train_np = np.array(A_train)
        
        history = model.fit(A_train_np, b_train, 
                            epochs = num_epoch, 
                            batch_size = num_batch_size, 
                            verbose = 1, 
                            validation_split=0.1)  

        # weights = model.layers[0].get_weights()[0]
        # # biases = model.layers[0].get_weights()[1]
        # print("The weights are :\n", weights)
        # print("The biases are  :", biases)
        
        
        
        
        true_data_train  = b_train
        pred_data_train = model.predict(A_train)
        
        true_data_test  = b_test
        pred_data_test  = model.predict(A_test)
        
        B_result_train_true = b_train
        B_result_train_pred = pred_data_train
        
        B_result_test_true = b_test
        B_result_test_pred = pred_data_test

        # Plot training & validation loss values
        print("model summary:",model.summary())

        # print(history.history.keys())
        plt.plot(history.history['loss'])
        plt.plot(history.history['val_loss'])
        plt.title('Model loss')
        plt.ylabel('Loss')
        plt.xlabel('Epoch')
        plt.legend(['Train', 'Test'], loc='upper left')
        plt.show()


    
   
    else:   # ----------- Linear regression -------------
        
        for i in range(B.shape[1]):  # B.shape[1]: number of features
    
            b = B[:,i]  # b = B[1], 2, 3, ... b = true value, to compare with pred value
            
        
            b_train = b[0:training_size]  # take all training value until training size
            A_train = A[0:training_size]
        
            # use second half test the model
            b_test = b[training_size: sample_size]   # take testing value
            A_test = A[training_size: sample_size]
        
            if scikit_LR_option == True:
                # print("Solving using Scikit-learn LR regression")
                # Solve using scikit-learn library
                # creating a regression model
                model = LinearRegression(n_jobs = -1, ) # use all processors
                # fitting the model
                model.fit(A_train, b_train)
                
                # making predictions
    
                true_data_train  = b_train
                pred_data_train  = model.predict(A_train) # make prediction on train set
    
                true_data_test  = b_test
                pred_data_test  = model.predict(A_test)  # make prediction on test set
    
                # print("model coefficients:\n", model.coef_)
                # print("model intercept:", model.intercept_)
                # print("model features seen:", model.n_features_in_)
                # print("rank:", model.rank_)
                # print("singular:", model.singular_)
                
            
            else:
                print("error perform linear regression")


        
            B_result_train_true[:,i] =   true_data_train   # put result in each column  # true/pred_data_xxx is use for plotting
            B_result_train_pred[:,i] =   pred_data_train
        
            B_result_test_true[:,i] =   true_data_test   # put result in each column
            B_result_test_pred[:,i] =   pred_data_test

    
    end_ML = timeit.default_timer()
    print('Time train %8.3f s'%(end_ML - start_ML))
    
    # define points to plot
    start_train_plt = start_train
    end_train_plt = end_train
    start_test_plt = start_test
    end_test_plt = end_test

    #  --------------------  performance metrics before scaled back data  --------------------------
    # print()
    # print("performance metric for TRAIN set (scaled data):")
    # performance_metrics(B_result_train_true, B_result_train_pred) # train set

    # print()
    # print("performance metric for TEST set (scaled data):")
    # performance_metrics(B_result_test_true, B_result_test_pred) # test set



    print()
    # transform back and plot
    if scaler_type == 'MinMax':
        print('Transform data back from MinMax scale')
        B_result_train_true = scalar.inverse_transform(B_result_train_true)
        B_result_train_pred = scalar.inverse_transform(B_result_train_pred)
        B_result_test_true = scalar.inverse_transform(B_result_test_true)
        B_result_test_pred = scalar.inverse_transform(B_result_test_pred)
        

    elif scaler_type == 'Std':
        print('Transform data back from Std scale')
        B_result_train_true = scalar.inverse_transform(B_result_train_true)
        B_result_train_pred = scalar.inverse_transform(B_result_train_pred)
        B_result_test_true = scalar.inverse_transform(B_result_test_true)
        B_result_test_pred = scalar.inverse_transform(B_result_test_pred)
        

    else:
        print('No scaled data')
        pass


    print()



    # ------------------- Plotting ------------------------------
    for i in range(B.shape[1]): # for range in number of features
        # plot training data 
        if plot_training_data == True:
            # print('Plotting training data is on')
            plot_train_data(B_result_train_true[:,i], B_result_train_pred[:,i], start_train_plt, end_train_plt,i ,column_name)
        else:
            # print('Plotting training data is off')
            pass
    
        # plot testing data
        if plot_testing_data == True:
            # print('Plotting testing data is on')
            plot_test_data(B_result_test_true[:,i], B_result_test_pred[:,i], start_test_plt, end_test_plt,i, column_name)
        else:
            # print('Plotting training data is off')
            pass
        
        
    
    
    B_result_train_true = np.array(B_result_train_true)  # convert back to array
    B_result_train_pred = np.array(B_result_train_pred)
    B_result_test_true = np.array(B_result_test_true)
    B_result_test_pred = np.array( B_result_test_pred)

    #  --------------------  performance metrics after scaled back data  --------------------------
    # print()
    # print("performance metric for TRAIN set (original scale data):")
    # number099_train = performance_metrics(B_result_train_true, B_result_train_pred) # train set

    print()
    print("performance metric for TEST set (original scale data):")
    number099_test = performance_metrics(B_result_test_true, B_result_test_pred) # test set

    
    # print("number099_train:", number099_train)
    # print("number099_test:", number099_test)
    # for x in number099_train:
    #     for i in x:
    #         print("those train column R^2 < 0.99 is:", column_name[i], "-->", i)

    if check_noise == True:
        for x in number099_test:
            for i in x:
                print("those test column R^2 < 0.99 is:", column_name[i], "-->", i)
    else:
        pass

    # -------------  performance metric and plot for front end of data ---------------
    # plot_front_end, plot_front_end_percentage, perform_metric_front_end
    # plot_partial_data(percentage, true_data_train, pred_data_train, iteration, title_name):

    print()
    
    if plot_front_end == True:
        print("B.shape[1]", B.shape[1])
        print("len col", len(column_name))
        for i in range(B.shape[1]):
        # plot training data 
                print(f"Plotting front {plot_front_end_percentage} % and end {plot_front_end_percentage} % of training data {i}:")
                # print('Plotting training data is on')
                plot_partial_data(plot_front_end_percentage, B_result_train_true[:,i], B_result_train_pred[:,i], i, column_name)
    
        # plot testing data

                print(f"Plotting front {plot_front_end_percentage} % and end {plot_front_end_percentage} % of testing data {i}:")
                # print('Plotting testing data is on')
                plot_partial_data(plot_front_end_percentage, B_result_test_true[:,i], B_result_test_pred[:,i], i, column_name)

    else:
        print("Not plotting front-end graph")
        pass


    
    if perform_metric_front_end == True:
        
        print("B_result_train_true.shape[0]: ",B_result_train_true.shape[0] )
        front_train =   B_result_train_true.shape[0] * performance_metric_front_end_percentage // 100
        print("front-percent metric: ",front_train )
        back_train  = - B_result_train_true.shape[0] * performance_metric_front_end_percentage //100  # count from back
        print("back-percent metric: ", back_train)

        print("B_result_test_true.shape[0]: ",B_result_test_true.shape[0] )
        front_test =   B_result_test_true.shape[0] * performance_metric_front_end_percentage // 100 # Floor division
        print("front-percent metric: ",front_test )
        back_test  = - B_result_test_true.shape[0] * performance_metric_front_end_percentage //100  # count from back
        print("back-percent metric: ", back_test)
        
        print()
        print(f"------performance metric for front {performance_metric_front_end_percentage} % TRAIN set-------:")
        print("B_result_train_true[:front_train] shape: ", B_result_train_true[:front_train].shape)
        performance_metrics(B_result_train_true[:front_train], B_result_train_pred[:front_train]) # train set
        print()
        print(f"------performance metric for back {performance_metric_front_end_percentage} % TRAIN set:-------")
        performance_metrics(B_result_train_true[back_train:], B_result_train_pred[back_train:]) # train set

        print()
        print("B_result_test_true[:front_test] shape: ", B_result_test_true[:front_test].shape)
        print(f"------performance metric for front {performance_metric_front_end_percentage} % TEST set:------")
        performance_metrics(B_result_test_true[:front_test], B_result_test_pred[:front_test]) # train set
        print()
        print(f"------performance metric for back {performance_metric_front_end_percentage} % TEST set:-------")
        performance_metrics(B_result_test_true[back_test:], B_result_test_pred[back_test:]) # train set
    else:
        print("Not calculating front end metrics")
        pass
        


    print("+++++++++++++++++++++++++++ END ++++++++++++++++++++++++++++++++")
    return B_result_train_true, B_result_train_pred, B_result_test_true, B_result_test_pred




In [4]:

# for plotting
import column_name as cn   # import column names form column_name.py file


sample_size10000 = 10000  # adjust how many data selected (end)
sample_size20000 = 20000

# ---------  choose molecule  ---------------------------------

molecule = 'H2-sto3g-10000'
# molecule = 'LiH-sto3g-10000'
# molecule = 'H2O-sto3g-10000'

# molecule = 'H2-sto3g-20000'
# molecule = 'LiH-sto3g-20000'
# molecule = 'H2O-sto3g-20000'

# molecule = 'H2O-631g-10000'
# molecule = 'LiH-631g-10000'

# molecule = 'LiH-sto3g-10000-y'
# molecule = 'LiH-sto3g-10000-x'

# ------------choose input parameters ----------------



kick_size = 1000   # how many data points to ignore   
# kick_size = 500
# remove_kick=True   # delta kick value is removed
remove_kick=False    # delta kick value is included

 # first and end performance metrics, and plot
# plot_front_end = True
plot_front_end = False
plot_front_end_percentage = 10

# performance_metric_front_end = True
performance_metric_front_end = False
performance_metric_front_end_percentage = 10
    
# scaler_type = 'MinMax'
scaler_type = 'Std'
# scaler_type = 'None'



# plot_training_data = True
plot_training_data = False 

# plot from where to where
start_train =  0      # plot for training set
end_train =    3000



# plot_testing_data = True
plot_testing_data = False 

start_test = 0        # plot for testing set
end_test =   3000

# check zeros in the dataset
# check_noise = True
check_noise = False

# filter_noise = True
filter_noise = False


# Using scikitlearn LinearRegression option
scikit_LR = True   # solve using scikit-learn LR method
# scikit_LR = False  

# Neural_Network = True   # use NN method
Neural_Network = False

# grid_search = True  # use grid search method
grid_search = False

 # -------------------------- input parameters -----------------------------------------

if molecule == 'H2-sto3g-10000' or molecule =='LiH-sto3g-10000' or molecule == 'H2O-sto3g-10000' or  molecule =='H2O-631g-10000' or  molecule =='LiH-631g-10000':
    sample_size = sample_size10000
elif molecule == 'H2-sto3g-20000' or molecule =='LiH-sto3g-20000' or molecule == 'H2O-sto3g-20000':
    sample_size = sample_size20000
else:
    print("error molecule size")

    

if molecule == 'H2-sto3g-10000' or molecule == 'H2-sto3g-20000':
    column_name = cn.column_name_H2
    print("column_name is H2:" )
elif molecule == 'LiH-sto3g-10000' or molecule == 'LiH-sto3g-20000':
    column_name = cn.column_name_LiH
    print("column_name is LiH:" )
elif molecule == 'H2O-sto3g-10000' or molecule == 'H2O-sto3g-20000':
    column_name = cn.column_name_H2O
    print("column_name is H2O:" )

elif molecule == 'H2O-631g-10000':
    column_name = cn.column_name_H2O_631g
    print("column_name is H2O-631g:" )
elif molecule == 'LiH-631g-10000':
    column_name = cn.column_name_LiH_631g
    print("column_name is LiH-631g:" )

else:
    print("error-column names")

# densitylist-Copy1(LiH)-sto3g-10000-perpendicular.csv
if molecule == 'LiH-sto3g-10000-y' or molecule == 'LiH-sto3g-10000-x':
    column_name = cn.column_name_LiH
    sample_size = sample_size10000
    # print("This is LiH-sto3g-10000-y, x" )

training_percentage = 50       #  adjust training percentage 

result_train_true, result_train_pred, result_test_true, result_test_pred = create_model(molecule, 
                      sample_size, 
                      training_percentage, 
                      scaler_type, 
                      remove_kick,
                      kick_size, 
                      plot_training_data, 
                      plot_testing_data,
                      start_train, end_train, start_test, end_test,
                      column_name,
                      plot_front_end,
                      plot_front_end_percentage,
                      performance_metric_front_end ,
                      performance_metric_front_end_percentage,
                      scikit_LR, check_noise, filter_noise, 
                      Neural_Network, grid_search)

# training_percentage2 = [10, 20, 30, 40, 50, 60, 70, 80, 90]    # loop through different percentage of train/test
# for tp in training_percentage2:
#     print(f"train {tp}%")
#     result_train_true, result_train_pred, result_test_true, result_test_pred = create_model(molecule, 
#                       sample_size, 
#                       tp, 
#                       scaler_type, 
#                       remove_kick,
#                       kick_size, 
#                       plot_training_data, 
#                       plot_testing_data,
#                       start_train, end_train, start_test, end_test,
#                       column_name,
#                       plot_front_end,
#                       plot_front_end_percentage,
#                       performance_metric_front_end ,
#                       performance_metric_front_end_percentage,
#                       scikit_LR, check_noise, filter_noise, Neural_Network,
#                       grid_search)

column_name is H2:
+++++++++++++++++++++++++++ START ++++++++++++++++++++++++++++++++
kick values are included.(full data)
train / test percentage:  50 % /  50 %
sample_size is: 10000
training_size is: 5000
test_size is: 5000
Current dataset: H2-sto3g-10000
Using Std Scaler

delta kick sample is included, take sample from first (whole dataset)
 B_result_train_true shape (5000, 4)
 B_result_train_pred shape (5000, 4)
 B_result_test_true shape (5000, 4)
 B_result_test_pred shape (5000, 4)
Plotting training data is off
Plotting testing data is off
B.shape[1] is number of dimension: 4

.............Solving using Scikit-learn LR regression..........
Time train    0.015 s

Transform data back from Std scale


performance metric for TEST set (original scale data):
R-squared (uniform average)  : 1.0000
R-squared (raw values)       : 
Number of R2 < 0.99:  0 / 4

Mean squared error (avg): 2.4945e-30
Mean squared error (raw):
Root mean squared error: 1.5794e-15
Mean absolute error (avg): 1.0280e

In [ ]:



# https://stats.stackexchange.com/questions/365545/why-are-machine-learning-algorithms-performing-worse-than-standard-multiple-line

In [ ]:
# https://youtu.be/bqBRET7tbiQ 155 - How many hidden layers and neurons do you need in your artificial neural network?
